# Chapter 9 — Routing and Coordination

Chapter 8 built a team. A team needs a front door. Real SOCs receive five hundred alerts
a day, of every kind, at every severity.

Routing has two stages, and confusing them is a security bug:

| Stage | Question | Who decides |
|---|---|---|
| Semantic | what *kind* of alert is this? | a model may guess |
| Severity | how *bad* is it? | policy — never a model |

A model that can downgrade severity is an attack surface with a friendly interface.

**Covered:** §9.2 semantic routing · §9.2.3 confidence and margin · §9.3.4 severity as
policy · §9.4 graceful degradation · §9.5.2 escalation with state · §9.6 MCP discovery.


## Setup

This lab installs from **one** `requirements.txt`.


In [ ]:
REPO_URL = "https://github.com/gstripling00/ai-engineer.git"

import os, sys, subprocess

if not os.path.isdir("aegis"):
    result = subprocess.run(["git", "clone", REPO_URL, "aegis"],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("git clone failed - check REPO_URL above.\n" + result.stderr)

os.chdir("aegis")
sys.path.insert(0, os.path.abspath("."))
print("repo:", os.getcwd())


In [ ]:
# --no-warn-conflicts silences a cosmetic Colab-only notice about `requests`;
# see the comment block at the top of requirements.txt. Real resolver errors still raise.
!pip -q install --no-warn-conflicts -r requirements.txt

Now verify the environment before running any lab code. This is the same check CI
runs, and it catches the one dependency conflict that would otherwise waste your
afternoon. It also confirms this chapter's source folder is in the checkout.


In [ ]:
!python tools/check_env.py --chapter 9

### Choosing a model tier

The labs read `AEGIS_MODEL` and swap the model behind a single seam:

| Tier | Cost | Determinism | Use it for |
|---|---|---|---|
| `mock` | free, no key | identical every run | learning the control flow; the test suite; CI |
| `openai` | billed per call | varies run to run | seeing a real model make these decisions |

Start on `mock`. Everything in this chapter runs there. When you switch to
`openai`, the code does not change — only the seam does.

Set the key from the environment, never as a literal in a cell. In Colab use the
key icon in the sidebar (Secrets); the cell below reads it without printing it.


In [ ]:
import os

os.environ["AEGIS_MODEL"] = "mock"     # free, deterministic, no key

# To use a real model instead, uncomment these two lines:
# from getpass import getpass
# os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: "); os.environ["AEGIS_MODEL"] = "openai"

print("model tier:", os.environ["AEGIS_MODEL"])


## Semantic routing

Match an alert to the handler whose description fits it best. Three of the four alerts
below route correctly. The fourth is a guess wearing a decision's clothes.


In [ ]:
import sys
sys.path.insert(0, "labs/chapter-09-routing-and-coordination")   # this chapter's source lives beside the notebook
from routing.router import semantic_route, ROUTE_DESCRIPTIONS

ALERTS = [
    {"id": "A-1", "rule": "Possible phishing email",
     "signals": ["suspicious link", "credential harvest"], "severity": "medium"},
    {"id": "A-2", "rule": "Brute force detected",
     "signals": ["repeated failed login", "account takeover"], "severity": "critical"},
    {"id": "A-3", "rule": "Large outbound transfer",
     "signals": ["data egress", "unusual destination"], "severity": "high"},
    {"id": "A-4", "rule": "Anomalous printer firmware update",
     "signals": ["unknown protocol", "no signature"], "severity": "low"},
]

print("handlers advertised:", list(ROUTE_DESCRIPTIONS))
print()
for alert in ALERTS:
    print(f'{alert["id"]}  {alert["rule"][:34]:36} -> {semantic_route(alert)}')
print()
print("A-4 is a printer firmware anomaly. It is not phishing, auth, or egress.")
print("The router had no idea - and routed anyway, because that is what a")
print("scoring function does. Nothing above tells you which routes it was sure of.")


## Confidence and margin

Now look at the numbers behind those decisions: the top score, and the **margin** over
the runner-up. Give the router permission to refuse.


In [ ]:
from routing.router import route_scores, route_with_confidence

print(f'{"alert":6} {"routed to":20} {"score":>7} {"margin":>8}')
for alert in ALERTS:
    s = route_scores(alert)
    print(f'{alert["id"]:6} {s["route"]:20} {s["score"]:7} {s["margin"]:8}')

print()
for alert in ALERTS:
    d = route_with_confidence(alert)
    if d["confident"]:
        print(f'{alert["id"]}  routed     -> {d["route"]}')
    else:
        print(f'{alert["id"]}  ESCALATED  -> {d["route"]}  ({d["reason"]}; '
              f'would_have_routed_to {d["would_have_routed_to"]})')
print()
print("The routing OUTPUT looked identical for all four. The score did not.")


## Severity is policy, not judgment

Severity decides whether a human sees the alert. That is a compliance commitment and a
staffing decision — not a judgment to delegate to a probabilistic system that can be
talked into things.

Ask what an attacker wants most from your routing layer. Not to be misclassified as
phishing instead of auth. They want `critical` to become `low`, so **nobody** looks.


In [ ]:
from routing.router import severity_route

for severity in ("low", "medium", "high", "critical"):
    print(f'{severity:9} -> {severity_route(severity)}')

runs = [severity_route("critical") for _ in range(50)]
print()
print("50 calls, distinct outcomes:", len(set(runs)))
print("deterministic:", len(set(runs)) == 1, "- a lookup table, in code, with no model in the path")


## Graceful degradation, flagged

Handlers go down. A router whose specialist is unavailable has two options: drop the
alert, or degrade to a generalist. Only one is acceptable — and it must be **recorded**,
because a generalist doing a specialist's job is doing it worse.

Graceful degradation that lies about itself is just a quieter failure.


In [ ]:
from routing.router import route_with_fallback

phishing = ALERTS[0]
print("all handlers up:", route_with_fallback(phishing))
print("phishing DOWN:  ", route_with_fallback(phishing,
                                              failed_routes={"phishing_handler"}))
print()
print("The alert was handled, not dropped - and the degradation is visible.")


## Escalation means handing over the work

When an alert goes to a human, *what* goes with it? Handing an analyst an id and a
severity means they start from zero, and you have automated only the notification.

The field people omit is **open questions** — what the agent could not determine. An
agent that reports its own limits is a colleague; one that does not is a noisy alarm.


In [ ]:
from routing.router import build_escalation

findings = {
    "severity": "critical",
    "verdict": "confirmed_compromise",
    "malicious_ip": True,
    "evidence": {"ip_verdict": "malicious", "egress_observed": True},
    "steps_taken": ["correlated auth failures", "checked IP reputation"],
    "open_questions": ["was any other account accessed from this IP?"],
}

escalation = build_escalation(ALERTS[1], findings)
import json
print(json.dumps(escalation, indent=2))


## MCP discovery: the router learns its own routes

Every router above had a **compiled-in** route table. Add a capability to the SOC and you
edit the router, review it, redeploy it.

Here the orchestrator asks an MCP server what it can do and builds the table from the
answer. A capability published this morning is routable this afternoon.

The cost, stated plainly: your route table now lives in a document you do not control —
which is exactly why Chapter 11 screens every tool description.


In [ ]:
from routing.mcp_discovery import (build_capability_server, discover_routes,
                                   route_via_discovery)

firmware_alert = {"id": "A-9", "rule": "Anomalous printer firmware update",
                  "signals": ["unsigned firmware", "unknown protocol",
                              "device management"]}

routes_before = discover_routes(build_capability_server())
print("routes discovered from the server:", sorted(routes_before))
before = route_via_discovery(firmware_alert, routes_before)
print("  unplanned alert ->", before["route"], f'(confident={before["confident"]})')

new_handler = {"name": "device_handler",
               "description": ("Handles device and firmware security: unsigned firmware, "
                               "printer, IoT, device management, unknown protocol.")}
routes_after = discover_routes(build_capability_server(extra_tools=[new_handler]))

print()
print("a handler is PUBLISHED SERVER-SIDE. client code changed: none.")
print("routes now:", sorted(routes_after))
after = route_via_discovery(firmware_alert, routes_after)
print("  same alert ->", after["route"], f'(confident={after["confident"]}, '
      f'score={after["score"]})')


---

## What you built

A routing front door: semantic classification, a confidence gate that can refuse,
degradation that admits it degraded, escalation carrying the work already done, and a
route table discovered at runtime.

- **A model may guess the kind. Only policy decides the severity.**
- **A router that cannot refuse will confidently misroute the one alert that mattered.**
- **Escalate the work, not the alarm.**


